In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
folder=r'Path'
OFolder=r'Output Folder Path'

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.xlsx') in f:
                    file_list.append(f)
file_list

In [5]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [6]:
len(file_link)

4

In [7]:
file_link[0].split("\\")[-1].split(".")[0]

'Centric'

In [ ]:
df_Match = pd.DataFrame(columns=['Brand', 'PartNumber', 'PartTerminologyName',  'PAName', 'Value'])
for i in range(len(file_link)):
    print(file_link[i])
    dfpadb=pd.read_excel(file_link[i],sheet_name="PiesPADBAttribute",skiprows=1)
    dfpadb=dfpadb[['Product','B15','PADB Name','Product Attribute']]#.drop_duplicates().reset_index(drop=True)
    dfa=pd.read_excel(file_link[i],sheet_name="PiesProductAttr",skiprows=1)
    dfa=dfa[['Product', 'B15','Attribute ID','Product Attribute']]#.drop_duplicates().reset_index(drop=True)
    dfpadb.rename(columns={'PADB Name': "PAName", 'Product': "PartTerminologyName",'B15':'PartNumber','Product Attribute':'Value'}, inplace=True)
    dfa.rename(columns={'Attribute ID': "PAName",'Product':'PartTerminologyName','B15':'PartNumber','Product Attribute':'Value'}, inplace=True)
    df_merged = pd.concat([dfa, dfpadb], ignore_index=True, sort=False)
    df_merged["Brand"]=file_link[i].split("\\")[-1].split(".")[0]
    df_Match=pd.concat([df_Match,df_merged], ignore_index=True, sort=False)

In [ ]:
df_Match

In [12]:
df_Match['PAName']=np.where(df_Match['PAName'].str.contains('[cntrc]'), df_Match['PAName'].str.replace('[cntrc]',""), df_Match['PAName'])

In [13]:
chunk_size=1000000
# Create a list of DataFrames by splitting the original DataFrame
df_chunks = [df_Match.iloc[i:i + chunk_size] for i in range(0, len(df_Match), chunk_size)]

In [14]:
df_Attributes=df_Match[['Brand', 'PartTerminologyName',  'PAName']].drop_duplicates().reset_index(drop=True)

In [15]:
with pd.ExcelWriter(OFolder+'\\'+'PartCat_Attribute_List.xlsx') as writer:  # doctest: +SKIP
    for i, chunk in enumerate(df_chunks):
        sheet_name = f"Chunk_{i+1}"  # Naming each sheet dynamically
        chunk.to_excel(writer, sheet_name=sheet_name, index=False)
    df_Attributes.to_excel(writer,index=False, sheet_name='FBG_Attributes')